# Complete RAGProbe Implementation

---
This implementation comes with the following improvments
- GPU Compute support
- Revised code base

**Importing Required Packages**

In [ ]:
import gzip
import json
from pathlib import Path

from datasets import load_dataset
from tqdm import tqdm

## C1: In-Built RAG Pipeline

**CONFIG**

In [ ]:
# dataset configuration
DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.en"
SPLIT = "train"

# Output location
OUTPUT_DIR = Path("../data/raw")
OUTPUT_FILE = OUTPUT_DIR / "wikipedia_articles.jsonl.gz"
MAX_ARTICLES = 10000

**Load wikipedia articles**

In [ ]:
# Streaming wikipedia articles

OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split = SPLIT, streaming = True)

In [ ]:
# saving each article as JSON object

article_count = 0
with gzip.open(OUTPUT_FILE, mode = 'wt', encoding = "utf-8") as output_file:
    for article in tqdm(dataset, desc = "Downloading articles"):
        if article_count >= MAX_ARTICLES:
            break

        article_id = str(article["id"])
        title = article["title"].strip()
        url = article["url"].strip()
        text = article["text"].strip()

        # skip empty or unusable records
        if not title or not text:
            continue

        record = {
            "id" : article_id,
            "title" : title,
            "url" : url,
            "text" : text
        }

        output_file.write(json.dumps(record, ensure_ascii = False) + "\n")
        article_count += 1
print(f"Saved to: {OUTPUT_FILE}")

In [ ]:
# Extracting the compressed JSONL file
import shutil

INPUT_FILE = Path("../data/raw/wikipedia_articles.jsonl.gz")
OUTPUT_FILE = Path("../data/raw/wikipedia_articles.jsonl")

with gzip.open(INPUT_FILE, "r") as compressed_file:
    with open(OUTPUT_FILE, "wb") as extracted_file:
        shutil.copyfileobj(compressed_file, extracted_file)

print(f"Extraction complete: {OUTPUT_FILE}")

In [ ]:
# Loading the articles into Document data structure manually
import json
from langchain_core.documents import Document

docs = []

with open("../data/raw/wikipedia_articles.jsonl",
          "r",
          encoding="utf-8") as f:

    for line in f:
        record = json.loads(line)

        docs.append(
            Document(
                page_content=record["text"],
                metadata={
                    "article_id": record["id"],
                    "title": record["title"],
                    "url": record["url"]
                }
            )
        )

**Chunking Documents**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200,
    length_function = len,
    separators = ["\n\n", "\n", ". ", " ", ""])
all_splits = text_splitter.split_documents(docs)
print("Number of chunks: ", len(all_splits))

**Generate Embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

# select GPU if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

texts = [chunk.page_content for chunk in all_splits]

#Load embedding model onto GPU
sen_trans = SentenceTransformer("all-MiniLM-L6-v2", device = device)

print("Embedding model dimensions: ", sen_trans.get_embedding_dimension())

# Generating embeddings
embeddings = sen_trans.encode(texts, 
                              show_progress_bar= True, 
                              normalize_embeddings= True,
                              batch_size = 32,
                              convert_to_numpy = True)
print("Embedding shape: ", embeddings.shape)

**Store Embeddings in VectorDB**

In [ ]:
# Importing required packages
import os
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid

# Configurations

collection_name = "Wikipedia_20231101_10k"
persist_directory = "../VectorStore"


# Create chromadb client and collection
os.makedirs(persist_directory, exist_ok = True)
client = chromadb.PersistentClient(path = persist_directory)

# Get or create collection
collection = client.get_or_create_collection(name = collection_name, metadata = {"Description": "HTML document embeddings for RAG"})
print("Exisiting documents in the collection: ", collection.count())

In [ ]:
# Prepare data for chromadb
ids = []
metadatas = []
chunk_text = []
embeddings_list = []

for i, (chunk, embedding) in enumerate(zip(all_splits, embeddings)):
    # Generate unique id:
    doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
    ids.append(doc_id)

    # Prepare metadata
    metadata = dict(chunk.metadata)
    metadata["doc_index"] = i
    metadata["content_length"] = len(chunk.page_content)
    metadatas.append(metadata)

    # append chunk content
    chunk_text.append(chunk.page_content)

    # append the corresponding embedding
    embeddings_list.append(embedding.tolist())


# --- CHUNKING & ADDING IN BATCHES ---
MAX_BATCH_SIZE = 5000  # Keeping it safely under the 5,461 limit

for i in range(0, len(ids), MAX_BATCH_SIZE):
    batch_ids = ids[i : i + MAX_BATCH_SIZE]
    batch_embeddings = embeddings_list[i : i + MAX_BATCH_SIZE]
    batch_metadatas = metadatas[i : i + MAX_BATCH_SIZE]
    batch_documents = chunk_text[i : i + MAX_BATCH_SIZE]

    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        documents=batch_documents,
    )
    print(
        f"Uploaded batch {i // MAX_BATCH_SIZE + 1}: Chunks {i} to {i + len(batch_ids)}"
    )


print(f"\nSuccessfully added {len(all_splits)} chunks to the vector store")
print(f"Total chunks in the collection: {collection.count()}")


**RAG retriver**

In [ ]:
"""
"""
This pipeline handles the query-based retrival from the vector-store

Required components

1. Vector Store: collection
2. Embedding generator: sen_trans

"""

# Retrieving Relevant documents for the query

query = "how can a nature of an appeal can vary?"
top_k = 5
# score_threshold = 0.0

# generating query embedding:
query_embedding = sen_trans.encode(query, normalize_embeddings=True)


# searching the vector store for relevant records
results = collection.query(query_embeddings= [query_embedding.tolist()], n_results= top_k)

if results['documents'] and results['documents'][0]:
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    ids = results['ids'][0]

retrieved_docs = []

# calculating the similarity score for each retrived documents
for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
    similarity_score = 1 - distance

    retrieved_docs.append({
        'id': doc_id,
        'document' : document,
        'metadata' : metadata,
        'similarity_score': similarity_score,
        'distance' : distance,
        'rank' : i + 1
    })

print(f"Number of documents retrieved after searching {len(retrieved_docs)}")

"""

**RAG Generator**

In [ ]:
"""# Importing Reuired Libraries
import os
from dotenv import load_dotenv
from openai import OpenAI

# loading the environment variables
load_dotenv()

# get the environment variable from .env file
Llama_3_1_8B_instruct_api_key = "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"

# Create a client
llm_client = OpenAI(
    api_key= Llama_3_1_8B_instruct_api_key,
    base_url = "https://openrouter.ai/api/v1"
)

# Building the context
context = "\n\n".join([doc['document'] for doc in retrieved_docs]) if retrieved_docs else ""

# Building the prompt
prompt = f"""Use the following context to answer the question concisely.
            context : {context}

            query: {query}

            Answer: """

# Getting the response from the model
response = llm_client.chat.completions.create(
    model = "nvidia/nemotron-3-ultra-550b-a55b:free",
    messages = [{
        "role" : "user",
        "content" : prompt
    }],
    temperature= 0.1,
    max_tokens= 1024
)
"""

## C2: Candidate Probe Generator

**CONFIG**

In [ ]:
import json
import random
import hashlib
import re
from collections import Counter
from pathlib import Path

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

BASELINE_TOP_K = TOP_K

# Supported interventions in the current local RAG implementation
SUPPORTED_CAPABILITIES = {
    "query_override",
    "top_k_override",
    "context_override"
}

# Initial retrieval configurations to test
RETRIEVAL_TOP_K_VALUES = [3, 8]

# One distractor per noise probe
NOISE_DOCUMENT_COUNT = 1

# Maximum query paraphrases generated per LLM request
PARAPHRASE_BATCH_SIZE = 6

In [ ]:
# checking if baseline and benchmark results correspond to each other

assert len(benchmark_cases) == len(baseline_results), (
    "Benchmark and baseline result counts do not match."
)

benchmark_by_id = {
    case["case_id"]: case
    for case in benchmark_cases
}

baseline_by_id = {
    result["case_id"]: result
    for result in baseline_results
}

assert set(benchmark_by_id) == set(baseline_by_id), (
    "Benchmark and baseline case IDs do not match."
)

print("Benchmark cases:", len(benchmark_by_id))
print("Baseline results:", len(baseline_by_id))
print("Baseline top_k:", BASELINE_TOP_K)

In [ ]:
# Defining candidate probe structure

def create_probe(
    base_case_id,
    family,
    subtype,
    severity,
    intervention_level,
    expected_contract,
    rationale,
    modified_query=None,
    retrieval_options=None,
    context_intervention=None,
    estimated_cost=1.0,
    seed=RANDOM_SEED
):

    retrieval_options = retrieval_options or {}
    context_intervention = context_intervention or {}

    fingerprint_data = {
        "base_case_id": base_case_id,
        "family": family,
        "subtype": subtype,
        "modified_query": modified_query,
        "retrieval_options": retrieval_options,
        "context_intervention": context_intervention,
        "seed": seed
    }

    canonical_data = json.dumps(
        fingerprint_data,
        sort_keys=True,
        ensure_ascii=False,
        default=str
    )

    fingerprint = hashlib.sha256(
        canonical_data.encode("utf-8")
    ).hexdigest()

    probe = {
        "probe_id": "probe_" + fingerprint[:12],
        "base_case_id": base_case_id,
        "family": family,
        "subtype": subtype,
        "severity": severity,
        "intervention_level": intervention_level,
        "modified_query": modified_query,
        "retrieval_options": retrieval_options,
        "context_intervention": context_intervention,
        "expected_contract": expected_contract,
        "estimated_cost": estimated_cost,
        "seed": seed,
        "fingerprint": fingerprint,
        "rationale": rationale,
        "validity_checks": {}
    }

    return probe

*Query Robustness Probes (Query Paraphraser)*

In [ ]:
def generate_paraphrases_batch(cases):

    input_questions = [
        {
            "case_id": case["case_id"],
            "question": case["query"]
        }
        for case in cases
    ]

    prompt = f"""
You are generating query paraphrases for a RAG reliability benchmark.

For each input question, generate exactly ONE paraphrase.

Rules:
1. Preserve the original meaning.
2. Preserve all named entities, dates and numerical values.
3. Preserve the requested answer type.
4. Do not introduce additional requirements.
5. Do not answer the question.
6. Do not add information that is not present in the original question.
7. Return the same case_id for each question.

Return ONLY a valid JSON array in this format:

[
    {{
        "case_id": "case_id",
        "paraphrased_question": "paraphrased question"
    }}
]

Input questions:

{json.dumps(input_questions, ensure_ascii=False)}
"""

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=2000
    )

    raw_output = response.choices[0].message.content.strip()

    # Remove markdown code fences if present
    raw_output = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_output,
        flags=re.IGNORECASE
    ).strip()

    generated = json.loads(raw_output)

    if isinstance(generated, dict):
        generated = generated.get("paraphrases", [])

    return generated

In [ ]:
# Run paraphrase generation in batches

# benchmark_cases : List

paraphrase_results = {}

for start in range(
    0,
    len(benchmark_cases),  #  <- pass only required number of cases here
    PARAPHRASE_BATCH_SIZE
):

    batch = benchmark_cases[
        start:start + PARAPHRASE_BATCH_SIZE
    ]

    print(
        f"Generating paraphrases for cases "
        f"{start + 1} to {start + len(batch)}"
    )

    generated = generate_paraphrases_batch(batch)

    for item in generated:

        case_id = item.get("case_id")
        paraphrase = item.get("paraphrased_question", "").strip()

        if case_id in benchmark_by_id and paraphrase:
            paraphrase_results[case_id] = paraphrase

print("\nGenerated paraphrases:", len(paraphrase_results))

In [ ]:
# Create one query probe per accepted paraphrase

query_probes = []

for case_id, paraphrase in paraphrase_results.items():

    probe = create_probe(
        base_case_id=case_id,
        family="query_robustness",
        subtype="llm_paraphrase",
        severity="low",
        intervention_level="query",

        modified_query=paraphrase,

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether a meaning-preserving change "
            "in question wording affects retrieval or answer quality."
        ),

        estimated_cost=1.0
    )

    query_probes.append(probe)

print("Query probes generated:", len(query_probes))

**Generate retrieval robustness probes**

In [ ]:
""""
We will test two alternative configurations:
- top_k = 3: tests whether retrieving fewer chunks removes useful evidence.
- top_k = 8: tests whether retrieving more chunks improves evidence availability or introduces additional noise.
"""

retrieval_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]

    for new_top_k in RETRIEVAL_TOP_K_VALUES:

        probe = create_probe(
            base_case_id=case_id,
            family="retrieval_robustness",
            subtype="top_k_variation",
            severity="low",
            intervention_level="retriever_config",

            retrieval_options={
                "top_k": new_top_k,
                "baseline_top_k": BASELINE_TOP_K
            },

            expected_contract="invariant_answer",

            rationale=(
                f"Tests the effect of changing top_k from "
                f"{BASELINE_TOP_K} to {new_top_k} on retrieval "
                "and downstream answer quality."
            ),

            estimated_cost=1.0
        )

        retrieval_probes.append(probe)

print("Retrieval probes generated:", len(retrieval_probes))

**Generate noise robustness probes**

In [ ]:
"""
This family tests whether irrelevant or weakly relevant passages influence the generated answer.

We will use the existing Wikipedia corpus to find candidate distractors. Specifically, we will:
1. Retrieve the top 20 chunks for a benchmark question.
2. Exclude the original top-five retrieved chunks.
3. Exclude gold evidence chunks.
4. Select one lower-ranked chunk as a distractor.
5. Append it to the original retrieved context.
"""


def get_candidate_distractor(case, baseline_result):

    query = case["query"]

    gold_ids = set(case.get("gold_evidence_ids", []))

    baseline_ids = set(baseline_result["retrieved_ids"])

    query_embedding = sen_trans.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=20,
        include=["documents", "metadatas", "distances"]
    )

    candidates = []

    for i, chunk_id in enumerate(results["ids"][0]):

        if chunk_id in gold_ids:
            continue

        if chunk_id in baseline_ids:
            continue

        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        if not document or len(document.strip()) < 200:
            continue

        candidates.append({
            "chunk_id": chunk_id,
            "text": document,
            "metadata": metadata,
            "distance": results["distances"][0][i]
        })

    # Prefer lower-ranked retrieved candidates
    if not candidates:
        return None

    return candidates[-1]

In [ ]:
# Generating noise probes

noise_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]
    baseline_result = baseline_by_id[case_id]

    distractor = get_candidate_distractor(
        case,
        baseline_result
    )

    if distractor is None:
        print("No distractor found for:", case_id)
        continue

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    context_with_noise = original_context + [distractor]

    probe = create_probe(
        base_case_id=case_id,
        family="noise_robustness",
        subtype="single_distractor_insertion",
        severity="low",
        intervention_level="context",

        context_intervention={
            "type": "append_distractor",
            "documents": context_with_noise,
            "original_document_ids": baseline_result["retrieved_ids"],
            "inserted_document_ids": [distractor["chunk_id"]],
            "insertion_count": 1
        },

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether an additional lower-ranked Wikipedia "
            "passage distracts the answer generator from the "
            "relevant evidence."
        ),

        estimated_cost=1.0
    )

    noise_probes.append(probe)

print("Noise probes generated:", len(noise_probes))

**Generate context-order robustness probes**

In [ ]:
"""
We will use the exact same documents retrieved during the baseline, but reverse their order.
Only the ordering changes. The number of documents, their IDs and their text remain identical.
"""

order_probes = []

for case in benchmark_cases:

    case_id = case["case_id"]
    baseline_result = baseline_by_id[case_id]

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    if len(original_context) < 2:
        continue

    reversed_context = list(reversed(original_context))

    probe = create_probe(
        base_case_id=case_id,
        family="context_order_robustness",
        subtype="reverse_order",
        severity="low",
        intervention_level="context",

        context_intervention={
            "type": "reorder_documents",
            "documents": reversed_context,
            "original_document_ids": baseline_result["retrieved_ids"],
            "permutation": "reverse"
        },

        expected_contract="invariant_answer",

        rationale=(
            "Tests whether changing only the order of retrieved "
            "documents changes the correctness or faithfulness "
            "of the generated answer."
        ),

        estimated_cost=1.0
    )

    order_probes.append(probe)

print("Context-order probes generated:", len(order_probes))

**Conflict robustness probes**

In [ ]:
# defining a function that accepts a manually verified fixture

def create_conflict_probe(case_id, fixture):

    required_fields = [
        "contradictory_text",
        "provenance",
        "verification_note"
    ]

    for field in required_fields:
        if not fixture.get(field):
            raise ValueError(f"Missing fixture field: {field}")

    if fixture.get("verified") is not True:
        raise ValueError(
            "Conflict fixture must be manually verified first."
        )

    case = benchmark_by_id[case_id]
    baseline_result = baseline_by_id[case_id]

    original_context = [
        {
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "distance": doc["distance"]
        }
        for doc in baseline_result["retrieved_documents"]
    ]

    conflict_document = {
        "chunk_id": "synthetic_conflict_" + case_id,
        "text": fixture["contradictory_text"],
        "metadata": {
            "source": "controlled_synthetic_intervention",
            "provenance": fixture["provenance"],
            "source_authority": fixture.get("source_authority")
        },
        "distance": None
    }

    context_with_conflict = original_context + [conflict_document]

    expected_contract = (
        "resolve_by_source_authority"
        if fixture.get("source_authority")
        else "recognize_conflict_or_abstain"
    )

    return create_probe(
        base_case_id=case_id,
        family="conflict_robustness",
        subtype="verified_contradiction",
        severity="medium",
        intervention_level="context",

        context_intervention={
            "type": "insert_contradiction",
            "documents": context_with_conflict,
            "inserted_document_ids": [
                conflict_document["chunk_id"]
            ],
            "provenance": fixture["provenance"],
            "source_authority": fixture.get("source_authority"),
            "verification_note": fixture["verification_note"]
        },

        expected_contract=expected_contract,

        rationale=(
            "Tests whether the RAG system appropriately handles "
            "explicitly verified contradictory evidence."
        ),

        estimated_cost=1.0
    )

**Combine all generated probes**

In [ ]:
candidate_probes = (
    query_probes
    + retrieval_probes
    + noise_probes
    + order_probes
)

print("Total candidates:", len(candidate_probes))

print("\nCandidates by family:")

family_counts = Counter(
    probe["family"]
    for probe in candidate_probes
)

for family, count in family_counts.items():
    print(f"{family}: {count}")

**Note :**
Conflict probes will be additional candidates once verified fixtures are supplied.

**Validate the candidate probe**

In [ ]:
MAX_QUERY_LENGTH = 2000
MAX_CONTEXT_CHARS = 30000

def validate_probe(probe, seen_fingerprints):

    errors = []

    required_fields = [
        "probe_id",
        "base_case_id",
        "family",
        "subtype",
        "severity",
        "intervention_level",
        "expected_contract",
        "fingerprint"
    ]

    # 1. Required fields
    for field in required_fields:
        if not probe.get(field):
            errors.append(f"Missing field: {field}")

    # 2. Base benchmark case
    if probe.get("base_case_id") not in benchmark_by_id:
        errors.append("Unknown base benchmark case")

    # 3. Expected contract
    if not probe.get("expected_contract"):
        errors.append("Missing expected behavioural contract")

    # 4. Duplicate detection
    if probe.get("fingerprint") in seen_fingerprints:
        errors.append("Duplicate candidate probe")

    # 5. Query intervention
    if probe["intervention_level"] == "query":

        modified_query = probe.get("modified_query")

        if not modified_query:
            errors.append("Modified query is empty")

        elif len(modified_query) > MAX_QUERY_LENGTH:
            errors.append("Modified query exceeds size limit")

        elif probe["base_case_id"] in benchmark_by_id:

            original_query = benchmark_by_id[
                probe["base_case_id"]
            ]["query"]

            if modified_query.strip() == original_query.strip():
                errors.append("Query was not modified")

    # 6. Retrieval configuration
    if probe["intervention_level"] == "retriever_config":

        options = probe.get("retrieval_options", {})
        new_top_k = options.get("top_k")

        if not isinstance(new_top_k, int) or new_top_k <= 0:
            errors.append("Invalid top_k configuration")

        elif new_top_k == BASELINE_TOP_K:
            errors.append("No change from baseline top_k")

        elif new_top_k > 20:
            errors.append("top_k exceeds configured limit")

    # 7. Context intervention
    if probe["intervention_level"] == "context":

        intervention = probe.get("context_intervention", {})
        documents = intervention.get("documents", [])

        if not documents:
            errors.append("Context intervention has no documents")

        else:

            total_chars = sum(
                len(doc.get("text", ""))
                for doc in documents
            )

            if total_chars > MAX_CONTEXT_CHARS:
                errors.append("Context exceeds configured size limit")

            if any(not doc.get("text") for doc in documents):
                errors.append("Context contains an empty document")

        # Noise-specific validation
        if probe["family"] == "noise_robustness":

            original_ids = set(
                intervention.get("original_document_ids", [])
            )

            inserted_ids = set(
                intervention.get("inserted_document_ids", [])
            )

            if not inserted_ids:
                errors.append("No distractor was inserted")

            if inserted_ids.intersection(original_ids):
                errors.append("Distractor duplicates original context")

        # Order-specific validation
        if probe["family"] == "context_order_robustness":

            original_ids = intervention.get(
                "original_document_ids", []
            )

            reordered_ids = [
                doc["chunk_id"]
                for doc in documents
            ]

            if reordered_ids != list(reversed(original_ids)):
                errors.append("Context permutation is incorrect")

    # 8. Capability validation
    required_capability = {
        "query": "query_override",
        "retriever_config": "top_k_override",
        "context": "context_override"
    }.get(probe["intervention_level"])

    if required_capability not in SUPPORTED_CAPABILITIES:
        errors.append("Unsupported intervention capability")

    # 9. Manual review requirements
    manual_review_required = probe["family"] in {
        "query_robustness",
        "noise_robustness",
        "conflict_robustness"
    }

    is_valid = len(errors) == 0

    probe["validity_checks"] = {
        "is_valid": is_valid,
        "errors": errors,
        "manual_review_required": manual_review_required,
        "manual_review_approved": not manual_review_required
    }

    if is_valid:
        seen_fingerprints.add(probe["fingerprint"])

    return probe

In [ ]:
# Run validation

seen_fingerprints = set()

validated_probes = []

for probe in candidate_probes:

    validated = validate_probe(
        probe,
        seen_fingerprints
    )

    validated_probes.append(validated)

In [ ]:
# Checking validation results

valid_probes = [
    p for p in validated_probes
    if p["validity_checks"]["is_valid"]
]

invalid_probes = [
    p for p in validated_probes
    if not p["validity_checks"]["is_valid"]
]

print("Total candidates:", len(validated_probes))
print("Valid candidates:", len(valid_probes))
print("Invalid candidates:", len(invalid_probes))

print("\nInvalid candidates:")

for probe in invalid_probes:
    print(
        probe["probe_id"],
        probe["validity_checks"]["errors"]
    )

In [ ]:
review_candidates = [
    p for p in valid_probes
    if p["validity_checks"]["manual_review_required"]
]

print("Candidates requiring review:", len(review_candidates))

for probe in review_candidates:

    case = benchmark_by_id[probe["base_case_id"]]

    print("\nProbe ID:", probe["probe_id"])
    print("Family:", probe["family"])
    print("Original question:", case["query"])

    if probe["family"] == "query_robustness":

        print(
            "Modified question:",
            probe["modified_query"]
        )

    elif probe["family"] == "noise_robustness":

        distractor = probe["context_intervention"]["documents"][-1]

        print("Distractor title:",
              distractor["metadata"].get("title"))

        print("Distractor text:",
              distractor["text"][:500])

    print("-" * 80)

In [ ]:
# approving all probes as of now

# Replace these with the IDs of candidates you actually reviewed.
approved_probe_ids = ["probe_47b93d943a6d", "probe_7d9ed9cb8663"]

for probe in valid_probes:

    if probe["probe_id"] in approved_probe_ids:

        probe["validity_checks"]["manual_review_approved"] = True

**Creating final Candidate Pool**

In [ ]:
eligible_probes = [
    probe
    for probe in validated_probes
    if (
        probe["validity_checks"]["is_valid"]
        and
        probe["validity_checks"]["manual_review_approved"]
    )
]

print("Total generated candidates:", len(candidate_probes))
print("Valid candidates:", len(valid_probes))
print("Eligible for selection:", len(eligible_probes))

print("\nEligible candidates by family:")

eligible_counts = Counter(
    probe["family"]
    for probe in eligible_probes
)

for family, count in eligible_counts.items():
    print(f"{family}: {count}")

**Saving the candidate Pool**

In [ ]:
CANDIDATE_DIR = Path("../data/candidates")
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

ALL_CANDIDATES_FILE = (
    CANDIDATE_DIR / "candidate_probes_v1.jsonl"
)

ELIGIBLE_CANDIDATES_FILE = (
    CANDIDATE_DIR / "eligible_probes_v1.jsonl"
)


def save_jsonl(records, filepath):

    with open(filepath, "w", encoding="utf-8") as file:

        for record in records:

            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )


save_jsonl(
    validated_probes,
    ALL_CANDIDATES_FILE
)

save_jsonl(
    eligible_probes,
    ELIGIBLE_CANDIDATES_FILE
)

print("All candidates saved:", ALL_CANDIDATES_FILE)
print("Eligible candidates saved:", ELIGIBLE_CANDIDATES_FILE)

--END--